# S&P 500 Historical Data: Exploratory Data Analysis & Preprocessing

**Quantitative Research Team — Hedge Fund Backtesting Project**  
**Authors:** Amon Ochuka & Aaron Ajwoga  

This notebook performs comprehensive Exploratory Data Analysis (EDA) on messy historical S&P 500 stock prices (`data/stock_prices.csv`) and the S&P 500 benchmark index (`data/sp500.csv`).

### Objectives:
1. **Memory Reduction Verification:** Validate optimized dtypes and memory footprint.
2. **Missing Values Analysis:** Quantify and visualize missing data patterns per variable and per year.
3. **Outliers Analysis:** Identify domain anomalies (spikes > $10k, zeros < $0.1, extreme monthly returns > 100% or < -50%).
4. **Historical Price Consistency:** Visualize average company price trends and cross-sectional consistency over time.
5. **Cross-Referenced Outlier Documentation:** Document extreme price corruptions against external financial sources (Yahoo Finance / SEC).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set working directory to project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebook' else os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

from memory_reducer import memory_reducer
from preprocessing import preprocessing

PLOTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"Project Root: {PROJECT_ROOT}")
print(f"Plots Directory: {PLOTS_DIR}")


## 1. Data Ingestion & Memory Optimization

We load `stock_prices.csv` and `sp500.csv` using `memory_reducer`, which downcasts integer and floating-point columns (ensuring floats are at minimum `np.float32`) to achieve:
- `stock_prices.csv` < 8.0 MB
- `sp500.csv` < 0.15 MB


In [ ]:
paths = {
    'prices': os.path.join(PROJECT_ROOT, 'data', 'stock_prices.csv'),
    'sp500': os.path.join(PROJECT_ROOT, 'data', 'sp500.csv')
}

raw_prices, raw_sp500 = memory_reducer(paths)
print(f"Prices shape: {raw_prices.shape}")
print(f"SP500 shape: {raw_sp500.shape}")


## 2. Missing Values Analysis

Equities in the S&P 500 naturally enter and leave the index over time (e.g., IPOs, mergers, acquisitions).
Here we analyze missing values per company and per year across the 2000–2014 observation window.


In [ ]:
def analyze_missing_values(df, date_col='date', plot=False, save_path=None):
    """Analyze and plot missing values per year with optional display."""
    df_temp = df.copy()
    df_temp[date_col] = pd.to_datetime(df_temp[date_col])
    df_temp['year'] = df_temp[date_col].dt.year
    
    price_cols = [c for c in df_temp.columns if c not in [date_col, 'year']]
    total_cells = len(df_temp) * len(price_cols)
    total_missing = df_temp[price_cols].isnull().sum().sum()
    
    print(f"Total price observations: {total_cells:,}")
    print(f"Total missing values: {total_missing:,} ({total_missing / total_cells:.2%})")
    
    # Missing values per year
    missing_by_year = df_temp.groupby('year')[price_cols].apply(lambda x: x.isnull().sum().sum())
    total_by_year = df_temp.groupby('year')[price_cols].apply(lambda x: x.size)
    missing_pct_by_year = (missing_by_year / total_by_year) * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_pct_by_year.plot(kind='bar', color='#3b82f6', edgecolor='black', ax=ax)
    ax.set_title('Missing Values Percentage per Year in S&P 500 Constituent Prices', fontsize=13, fontweight='bold')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Missing Data (% of cells)', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"Saved missing values plot to {save_path}")
    
    if plot:
        plt.show()
    else:
        plt.close()
        
    return missing_pct_by_year

missing_plot_file = os.path.join(PLOTS_DIR, 'missing_values_per_year.png')
missing_stats = analyze_missing_values(raw_prices, plot=False, save_path=missing_plot_file)
print("\nMissing percentage by year:")
print(missing_stats.round(2))


## 3. Outlier Analysis & Extreme Price Spikes

Real-world financial data feeds frequently suffer from:
1. **Decimal shifts / Multiplier corruptions** (e.g., $11.72 recorded as $11,725,800.00).
2. **Zero prices** ($0.00 for active companies).
3. **Severe single-day spikes** exceeding $10,000/share.

Here we detect all prices outside the valid operational boundary: `$0.10 <= Price <= $10,000.00`.


In [ ]:
def detect_price_outliers(df, date_col='date', min_price=0.1, max_price=10000.0):
    """Detect price outliers outside valid range."""
    price_cols = [c for c in df.columns if c != date_col]
    outliers_list = []
    
    for col in price_cols:
        series = df[[date_col, col]].dropna()
        too_high = series[series[col] > max_price]
        too_low = series[series[col] < min_price]
        
        for _, r in too_high.iterrows():
            outliers_list.append({'ticker': col, 'date': r[date_col], 'price': r[col], 'type': 'High Spike (>10k)'})
        for _, r in too_low.iterrows():
            outliers_list.append({'ticker': col, 'date': r[date_col], 'price': r[col], 'type': 'Zero/Low (<0.1)'})
            
    outliers_df = pd.DataFrame(outliers_list)
    return outliers_df

outliers_df = detect_price_outliers(raw_prices)
print(f"Total anomalous price points found: {len(outliers_df):,}")
print("\nTop 10 Extreme Price Spikes (> $10,000):")
high_spikes = outliers_df[outliers_df['type'] == 'High Spike (>10k)'].sort_values(by='price', ascending=False)
print(high_spikes.head(10))


## 4. Price Consistency & Average Price Graph

We compare average stock price over time before vs after outlier filtering.
Without outlier correction, a single corrupted price (e.g. $11.7 million) distorts the aggregate market mean by orders of magnitude.


In [ ]:
def plot_average_price_over_time(df, date_col='date', plot=False, save_path=None):
    """Compute and plot average company price over time comparing raw vs cleaned."""
    df_temp = df.copy()
    df_temp[date_col] = pd.to_datetime(df_temp[date_col])
    df_temp = df_temp.set_index(date_col).sort_index()
    price_cols = [c for c in df_temp.columns]
    
    # Raw mean across constituents per date
    raw_daily_mean = df_temp[price_cols].mean(axis=1)
    
    # Filtered mean (prices between 0.1 and 10000)
    cleaned_df = df_temp[price_cols].where((df_temp[price_cols] >= 0.1) & (df_temp[price_cols] <= 10000.0))
    cleaned_daily_mean = cleaned_df.mean(axis=1)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    ax1.plot(raw_daily_mean.index, raw_daily_mean.values, color='#ef4444', linewidth=1.2)
    ax1.set_title('Raw Average S&P 500 Constituent Price Over Time (Corrupted by Spikes)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Raw Mean Price ($)', fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    ax2.plot(cleaned_daily_mean.index, cleaned_daily_mean.values, color='#10b981', linewidth=1.5)
    ax2.set_title('Cleaned Average S&P 500 Constituent Price Over Time ($0.10 <= P <= $10,000)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Date', fontsize=11)
    ax2.set_ylabel('Cleaned Mean Price ($)', fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"Saved average price plot to {save_path}")
        
    if plot:
        plt.show()
    else:
        plt.close()
        
    return cleaned_daily_mean

avg_price_plot_file = os.path.join(PLOTS_DIR, 'average_price_over_time.png')
cleaned_mean = plot_average_price_over_time(raw_prices, plot=False, save_path=avg_price_plot_file)


## 5. Cross-Referenced Outliers Documentation

As required, we document at least 5 identified outliers cross-referenced against historical market data (Yahoo Finance, SEC filings):
- **SPLS (2000-12-12)**: Reported $11,725,800.00 vs Actual ~$11.75
- **CSC (2014-12-30)**: Reported $611,308.01 vs Actual ~$64.25
- **STT (2014-12-26)**: Reported $593,809.01 vs Actual ~$78.50
- **TMK (2014-12-30)**: Reported $483,954.38 vs Actual ~$54.10
- **SWY (2014-12-19)**: Reported $104,693.31 vs Actual ~$34.50

This information is saved to `results/outliers.txt`.


In [ ]:
# Run preprocessing pipeline on raw data
processed_prices, processed_sp500 = preprocessing(raw_prices, raw_sp500)
print(f"Processed Prices MultiIndex Shape: {processed_prices.shape}")
print(f"Processed SP500 Shape: {processed_sp500.shape}")
print("\nProcessed Prices Head:")
display(processed_prices.head())
